# Compare scaling decisions using measurement fixtures

A classifier needs a reproducible scaling report. Use a serial timing trace, worker summaries, request arrivals and served latencies to compare candidate resource decisions.

Fill in each starter, then run its check cell. Restart the kernel and run all cells before submitting. Keep your interpretation in the last Markdown cell.


## 1. Estimate serial workload speedup

Calculate the effect of speeding up one stage.

Implement serial_speedup(unchanged_ms, improved_ms, factor) for finite nonnegative times totalling more than zero and a finite positive factor. Reject invalid times or nonpositive factors.


In [ ]:
def serial_speedup(unchanged_ms, improved_ms, factor):
    raise NotImplementedError


In [ ]:
assert serial_speedup(60,40,2) == 1.25
assert serial_speedup(0,10,2) == 2
assert serial_speedup(10,0,5) == 1
assert serial_speedup(5,5,0.5) < 1
for args in [(0,0,1),(-1,2,1),(2,1,0)]:
    try:
        serial_speedup(*args)
    except ValueError:
        pass
    else:
        raise AssertionError("reject invalid input")
print("OK")


## 2. Recover a global per-example gradient

Reduce local means with per-example weights.

Implement global_gradient(rows) for a list of (nonnegative count, finite gradient mean vector) pairs. Reject negative counts and unequal vector lengths. Return None for no examples.


In [ ]:
def global_gradient(rows):
    raise NotImplementedError


In [ ]:
assert global_gradient([(1,[0,8]),(3,[4,0])]) == [3,2]
assert global_gradient([(0,[9]),(2,[4])]) == [4]
assert global_gradient([]) is None
assert global_gradient([(0,[1])]) is None
for rows in ([(1,[0]),(1,[0,1])],[(-1,[0])]):
    try:
        global_gradient(rows)
    except ValueError:
        pass
    else:
        raise AssertionError("reject inconsistent summaries")
print("OK")


## 3. Schedule simple dynamic batches

Schedule dynamic batches under a size and wait bound.

Implement dispatch_batches(arrivals, capacity, delay) for a sorted list of millisecond arrival times. Return (launch_time, batch_arrivals) pairs. Launch at the final arrival when full or at the oldest deadline otherwise. Reject nonpositive capacity and negative delay.


In [ ]:
def dispatch_batches(arrivals, capacity, delay):
    raise NotImplementedError


In [ ]:
assert dispatch_batches([0,2,3,10],3,4) == [(3,[0,2,3]),(14,[10])]
assert dispatch_batches([0,4,5],3,4) == [(4,[0,4]),(9,[5])]
assert dispatch_batches([0,0,1],2,0) == [(0,[0,0]),(1,[1])]
assert dispatch_batches([],1,1) == []
for args in [([1],0,1),([],1,-1)]:
    try:
        dispatch_batches(*args)
    except ValueError:
        pass
    else:
        raise AssertionError("reject nonpositive capacity and negative delay")
print("OK")


## 4. Summarise one load window

Report latency percentiles, failures and on-time completions.

Implement load_summary(latencies, target_ms) for a list of successful millisecond latencies and None for failures. Return attempted, failed, nearest-rank p95_success_ms or None, and number of successes at or below target.


In [ ]:
def load_summary(latencies, target_ms):
    raise NotImplementedError


In [ ]:
assert load_summary([15,None,60,20],20) == {"attempted":4,"failed":1,"p95_success_ms":60,"on_time":2}
assert load_summary([None],10) == {"attempted":1,"failed":1,"p95_success_ms":None,"on_time":0}
assert load_summary(list(range(1,21)),19)["p95_success_ms"] == 19
assert load_summary(list(range(1,12)),0)["p95_success_ms"] == 11
assert load_summary([],5)["attempted"] == 0
print("OK")


## 5. Compare simulated batching policies with the same arrivals

Implement batching_report(arrivals, capacity, delay, service_ms, target_ms). Use dispatch_batches
and assume each launched batch finishes service_ms later with no interference from other batches.
For each request compute launch minus arrival plus service_ms. Pass those times to load_summary.
Return (batch count, load summary). This is a queue-delay accounting fixture. It does not model a
single busy accelerator or determine real capacity.


In [ ]:
def batching_report(arrivals, capacity, delay, service_ms, target_ms):
    raise NotImplementedError


In [ ]:
assert batching_report([0,2,3,10],3,4,5,8) == (2,{"attempted":4,"failed":0,"p95_success_ms":9,"on_time":3})
assert batching_report([0,2,3,10],1,4,5,8) == (4,{"attempted":4,"failed":0,"p95_success_ms":5,"on_time":4})
assert batching_report([],3,4,5,8) == (0,{"attempted":0,"failed":0,"p95_success_ms":None,"on_time":0})
print("OK")


## Interpretation

Choose whether to add workers, optimise ingestion or change a batch deadline. State the measurement that would justify the choice and a controlled comparison you would run next.

**Your answer**
